# 팀x팀 맞대결 + 시즌 보정 피처 실험 노트북

`train_ensemble.py`에 새로 추가한 두 피처가 LightGBM 성능을 개선하는지 확인합니다.
한 번에 같이 넣고, feature importance로 각각의 기여도를 따로 확인합니다.

**1) `asof_team_matchup_success_rate`** (팀x팀 as-of 맞대결)
- 사전 검증: 96개 조합, 조합당 평균 15,365행(최소 292) — 실패했던 개인 맞대결(96,133개 조합,
  조합당 평균 15행)과 완전히 다른 표본 규모. 성공했던 팀 단위 피처와 유사한 이유로 기대.
- 상관계수 0.0488 (참고 - 팀 단위 단독 0.0422)

**2) 시즌 보정 (`pitcher_rate_vs_season`, `batter_rate_vs_season`)**
- 사전 검증: 시즌별 성공률이 2019 0.565 -> 2024 0.486로 7.9%p 하락하는 큰 추세 확인.
  `asof_pitcher_success_rate`는 커리어 누적이라 이 추세가 섞여있어, 시즌 as-of 기준선을
  빼서 선수 개인의 순수 편차를 분리해봄.
- 상관계수: `pitcher_rate_vs_season` 0.0670, `batter_rate_vs_season` 0.0373

**비교 기준선** (상성+팀 피처까지 반영된 최신 값, `randomforest.ipynb` 최근 재실행 결과):
- LightGBM 전체(147만행) OOF Brier: **0.243817**

**이 파일과 `train_ensemble.py`는 같은 폴더에 있어야 아래 import가 동작합니다.**

In [1]:
import sys, os, time
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

sys.path.append(os.getcwd())
from train_ensemble import (
    TARGET_COL, CAT_COLS, build_features, train_lgb,
)

DATA_DIR = "../open/data"

## 1. 데이터 로드 & 피처 생성 (팀x팀 맞대결 + 시즌 보정 포함)

In [2]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")
print(train.shape)

train_feat, feat_cols = build_features(
    train, None, use_team_matchup_feature=True, use_season_trend_feature=True
)
cat_features = [c for c in CAT_COLS if c in feat_cols]

new_cols = [c for c in feat_cols if c in
            ("asof_team_matchup_n", "asof_team_matchup_success_rate",
             "asof_season_success_rate", "pitcher_rate_vs_season", "batter_rate_vs_season")]
print(f"피처 개수: {len(feat_cols)} (신규: {new_cols})")

r = train_feat[TARGET_COL].mean()
baseline_brier = r * (1 - r)
print(f"기준(무정보) Brier = {baseline_brier:.5f}")

(1475092, 49)
피처 개수: 80 (신규: ['asof_team_matchup_n', 'asof_team_matchup_success_rate', 'asof_season_success_rate', 'pitcher_rate_vs_season', 'batter_rate_vs_season'])
기준(무정보) Brier = 0.24944


## 2. 전체 데이터로 바로 확인

지금까지의 경험상 20만행 결과는 신뢰도가 낮으므로 바로 전체 데이터로 확인합니다.

In [3]:
X_full = train_feat[feat_cols]
y_full = train_feat[TARGET_COL].values

t0 = time.time()
lgb_models_new, lgb_oof_new = train_lgb(X_full, y_full, X_full, cat_features)
brier_new = brier_score_loss(y_full, lgb_oof_new)
print(f"[LightGBM+신규2종] 소요시간: {time.time()-t0:.1f}초")
print(f"[LightGBM+신규2종] OOF Brier (전체): {brier_new:.5f}")
print(f"참고 - 신규 피처 없는 LightGBM 전체 Brier: 0.243817")
print(f"개선폭: {(0.243817 - brier_new) / 0.243817 * 100:.4f}% (양수면 개선)")

  [LGB fold 0] brier=0.24372
  [LGB fold 1] brier=0.24376
  [LGB fold 2] brier=0.24373
  [LGB fold 3] brier=0.24382
  [LGB fold 4] brier=0.24378
[LightGBM+신규2종] 소요시간: 261.6초
[LightGBM+신규2종] OOF Brier (전체): 0.24376
참고 - 신규 피처 없는 LightGBM 전체 Brier: 0.243817
개선폭: 0.0237% (양수면 개선)


## 3. Feature Importance로 각 피처 기여도 확인

In [4]:
imp_df = pd.DataFrame({
    f"fold{i}": m.feature_importance(importance_type="gain")
    for i, m in enumerate(lgb_models_new)
}, index=feat_cols)
imp_df["mean_gain"] = imp_df[[c for c in imp_df.columns if c.startswith("fold")]].mean(axis=1)
imp_df["share_pct"] = imp_df["mean_gain"] / imp_df["mean_gain"].sum() * 100
imp_df = imp_df.sort_values("mean_gain", ascending=False)

imp_df_ranked = imp_df.reset_index().rename(columns={"index": "feature"})
imp_df_ranked["rank"] = imp_df_ranked.index + 1
print("신규 피처 순위:")
display(imp_df_ranked[imp_df_ranked["feature"].isin(new_cols)][["rank", "feature", "mean_gain", "share_pct"]])

신규 피처 순위:


,rank,feature,mean_gain,share_pct
2,3,asof_season_success_rate,39996.718456,3.428449
3,4,asof_team_matchup_success_rate,38933.429124,3.337306
17,18,batter_rate_vs_season,25560.658241,2.191015
29,30,asof_team_matchup_n,23498.364082,2.014239
31,32,pitcher_rate_vs_season,20146.845325,1.726953
